In [1]:
# 1. Cài đặt các thư viện lõi
!pip install filterpy lapx motmetrics cython_bbox opencv-python-headless loguru easydict scipy yacs faiss-cpu pandas matplotlib

# 2. Clone repo BoT-SORT
!git clone https://github.com/NirAharon/BoT-SORT.git

# 3. Tạo thư mục 'pretrained' BÊN TRONG BoT-SORT theo đúng tài liệu
!mkdir -p /kaggle/working/BoT-SORT/pretrained

# 4. Tải mô hình ReID (mot17_sbs_S50.pth) vào đúng thư mục vừa tạo

!gdown "1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN" -O /kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.2 MB/s eta 0:00:00:00:0100:01
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=40db8107a956e58775d0168c7be8cd62655cc6285d9152d0fed10f00252b344e
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
  Created wheel for cython_bbox: filename=cython_bbox-0.1.5-cp312-cp312-linux_x86_64.whl size=111559 sha256=ec64120c26435da544a39cd7b959ba32a3d202c48dbcf8851707c5f5f9a7c122
  Stored in directory: /root/.cache/pip/wheels/f1/e7/0a/7c310ac8921f2c1

In [2]:
import sys
import os
import glob
import cv2
import pandas as pd
import numpy as np
import torch
import collections
import collections.abc
import types
import motmetrics as mm
from tqdm.notebook import tqdm

# Nạp BoT-SORT và FastReID vào hệ thống ---
BOT_SORT_PATH = '/kaggle/working/BoT-SORT'
sys.path.insert(0, BOT_SORT_PATH) 
sys.path.insert(0, os.path.join(BOT_SORT_PATH, 'fast_reid'))

In [3]:
# --- FIX LỖI NUMPY 2.0 ---
# Định nghĩa lại hàm asfarray đã bị xóa bằng cách trỏ nó về asarray(..., dtype=float)
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda x: np.asarray(x, dtype=float)
# -------------------------

# 1. Vá lỗi Python 3.10+ (ImportError: cannot import name 'Mapping'...)
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping

# 2. Vá lỗi PyTorch 2.0+ (ModuleNotFoundError: No module named 'torch._six')
# Code cũ tìm 'string_classes' trong 'torch._six', ta tạo giả nó.
if 'torch._six' not in sys.modules:
    dummy_six = types.ModuleType('torch._six')
    dummy_six.string_classes = str
    sys.modules['torch._six'] = dummy_six
    # Gán ngược vào torch để chắc chắn
    if not hasattr(torch, '_six'):
        torch._six = dummy_six

# --- VÁ LỖI NUMPY (Chạy 1 lần là được) ---
# Khôi phục lại các thuộc tính đã bị xóa ở Numpy bản mới
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'bool'):
    np.bool = bool

print("Đã vá lỗi np.float thành công!")

Đã vá lỗi np.float thành công!


In [4]:
from tracker.bot_sort import BoTSORT

class BoTSORTArgs:
    def __init__(self):
        self.track_high_thresh = 0.5
        self.track_low_thresh = 0.2
        self.new_track_thresh = 0.7
        self.match_thresh = 0.8
        self.track_buffer = 30
        self.mot20 = False
        self.cmc_method = "none" 
        self.name = "bot_sort"
        self.ablation = False 
        self.fps = 30

        self.with_reid = True
        # File config của mạng SBS-S50 có sẵn trong thư mục clone
        self.fast_reid_config = r"/kaggle/working/BoT-SORT/fast_reid/configs/MOT17/sbs_S50.yml"
        # Trọng số vừa tải vào thư mục pretrained nội bộ
        self.fast_reid_weights = r"/kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth"
        self.proximity_thresh = 0.5
        self.appearance_thresh = 0.25
        self.aspect_ratio_thresh = 100.0
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

2026-03-15 13:32:49.059441: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773581569.252131      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773581569.313062      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773581569.755172      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773581569.755219      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773581569.755222      55 computation_placer.cc:177] computation placer alr

# Hàm bổ trợ

In [5]:
def parse_mot17_det(det_path):
    columns = ['frame','track_id','x','y','w','h','conf','x3','y3','z3']
    df = pd.read_csv(det_path, header=None, names=columns)

    df['x2'] = df['x'] + df['w']
    df['y2'] = df['y'] + df['h']

    detections = {}

    for frame_idx, group in df.groupby('frame'):
        frame_dets = []
        for _, row in group.iterrows():

            frame_dets.append({
                'boxs': [row['x'], row['y'], row['x2'], row['y2']],
                'score': row['conf'],
                'label': 1
            })

        detections[int(frame_idx)-1] = frame_dets

    return detections

def parse_mot17_gt(gt_path):

    cols = ['frame','track_id','x','y','w','h','conf','class','vis']
    df = pd.read_csv(gt_path, header=None, names=cols)

    # chỉ giữ pedestrian
    df = df[df['class'] == 1]

    # bỏ object invisible
    df = df[df['vis'] > 0]

    df['frame'] = df['frame'] - 1

    return df

# Try w 1 seq

In [6]:
# img_dir = "/kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/train/MOT17-02-SDP/img1"
# det_file = "/kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/train/MOT17-02-SDP/det/det.txt"
# gt_file = "/kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/train/MOT17-02-SDP/gt/gt.txt"

# # 1. Load dâta
# print(f"Loading data for Sequence ")
# # seq_detections, gt_df = parse_mot17_line_by_line(label_file)
# seq_detections = parse_mot17_det(det_file)
# gt_df = parse_mot17_gt(gt_file)
# image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

# if len(image_paths) == 0:
#     print("❌ Lỗi: Không tìm thấy ảnh. Hãy kiểm tra lại đường dẫn KAGGLE_ROOT")
# else:
#     print(f"✅ Đã tải {len(image_paths)} frames.")

# # 2. Khởi tạo metrics
# acc = mm.MOTAccumulator(auto_id=True)

# # 3. KHỞI TẠO TRACKER
# # TAU_HIGH = 0.6
# # TAU_LOW = 0.1
# MAX_LOST_FRAMES = 30 # số frames giữ tracker bị mất trước khi xóa hẳn

# tracker_list = []
# track_id_count = 1
# motion_estimator = CameraMotionEstimator()

# print("\n>>> Start Tracking...")

# for k, img_path in enumerate(tqdm(image_paths)):
#     img = cv2.imread(img_path)
#     if img is None:
#         continue

#     # =========================
#     # 1. GET DETECTIONS
#     # =========================
#     frame_dets = seq_detections.get(k, [])

#     # =========================
#     # 2. PREDICT + CAMERA MOTION
#     # =========================
#     M = motion_estimator.estimate(img)
#     for trk in tracker_list:
#         trk.predict()
#         trk.apply_camera_motion(M)

#     # =========================
#     # 3. ASSOCIATION
#     # =========================
#     matches, unmatched_dets, unmatched_trks = associate_object(
#         detections=frame_dets,
#         trackers=tracker_list,
#         current_image=img
#     )

#     # =========================
#     # 4. UPDATE MATCHED
#     # =========================
#     matched_trk_ids = set()
#     for trk_idx, det_idx in matches:
#         trk = tracker_list[trk_idx]
#         det = frame_dets[det_idx]
#         trk.update(det['boxs'], raw_det=det, image=img)
#         matched_trk_ids.add(trk.id)

#     # =========================
#     # 5. CREATE NEW TRACKS
#     # =========================
#     for det_idx in unmatched_dets:
#         det = frame_dets[det_idx]
#         new_trk = Tracker(track_id_count, det['boxs'], det['label'], initial_image=img)
#         tracker_list.append(new_trk)
#         track_id_count += 1

#     # =========================
#     # 6. REMOVE DEAD TRACKS
#     # =========================
#     tracker_list = [t for t in tracker_list if t.time_since_update <= MAX_LOST_FRAMES]

#     # =========================
#     # 7. COLLECT HYPOTHESIS (ONLY MATCHED)
#     # =========================
#     frame_hypos = []
#     frame_hypo_ids = []

#     for trk in tracker_list:
#         if trk.time_since_update == 0:  # chỉ lấy tracker vừa match
#             box = get_box_coords(trk.kf.x.flatten()[:4])

#             x1, y1, x2, y2 = box
            
#             w = x2 - x1
#             h = y2 - y1
#             frame_hypos.append([x1, y1, w, h])
#             frame_hypo_ids.append(trk.id)

#     # =========================
#     # 8. COLLECT GT
#     # =========================
#     frame_gt = []
#     frame_gt_ids = []

#     if not gt_df.empty:
#         gt_now = gt_df[gt_df['frame'] == k]
#         for _, row in gt_now.iterrows():
#             if int(row['track_id']) == -1:
#                 continue
#             # w = row['bbox_r'] - row['bbox_l']
#             # h = row['bbox_b'] - row['bbox_t']
#             w = row['w']
#             h = row['h']
#             frame_gt.append([row['x'], row['y'], w, h])
#             frame_gt_ids.append(int(row['track_id']))

#     # =========================
#     # 9. UPDATE METRICS
#     # =========================
#     if not hasattr(np, 'asfarray'):
#         np.asfarray = lambda x: np.asarray(x, dtype=float)

#     dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
#     acc.update(frame_gt_ids, frame_hypo_ids, dist_matrix)

# print("Tracking Completed.")

In [7]:
# # print(f"\n{'='*20} EVALUATION REPORT FOR SEQ {'='*20}")

# # mh = mm.metrics.create()
# # # Các chỉ số quan trọng: MOTA, IDF1 (độ ổn định ID), MOTP (độ khớp box)
# # summary = mh.compute(acc, metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'], name='OriginalAssociation')

# # strsummary = mm.io.render_summary(
# #     summary,
# #     formatters=mh.formatters,
# #     namemap=mm.io.motchallenge_metric_names
# # )
# # print(strsummary)

# mh = mm.metrics.create()

# summary = mh.compute(
#     acc,
#     metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'],
#     name='MOT17-05'
# )

# print(mm.io.render_summary(summary))

# Scale lên toàn bộ data

In [8]:
import os
import glob

DATA_ROOT = "/kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/train"

sequences = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])
# sequences = sequences[:2]
print("Sequences:", sequences)

Sequences: ['MOT17-02-DPM', 'MOT17-02-FRCNN', 'MOT17-02-SDP', 'MOT17-04-DPM', 'MOT17-04-FRCNN', 'MOT17-04-SDP', 'MOT17-05-DPM', 'MOT17-05-FRCNN', 'MOT17-05-SDP', 'MOT17-09-DPM', 'MOT17-09-FRCNN', 'MOT17-09-SDP', 'MOT17-10-DPM', 'MOT17-10-FRCNN', 'MOT17-10-SDP', 'MOT17-11-DPM', 'MOT17-11-FRCNN', 'MOT17-11-SDP', 'MOT17-13-DPM', 'MOT17-13-FRCNN', 'MOT17-13-SDP']


In [9]:
import time
args = BoTSORTArgs()
accs = []
names = []

CONF_THRESH = 0.2
tracking_time = 0
tracking_calls = 0

for seq in sequences:

    print(f"\n========== Running {seq} ==========")

    img_dir = os.path.join(DATA_ROOT, seq, "img1")
    det_file = os.path.join(DATA_ROOT, seq, "det/det.txt")
    gt_file = os.path.join(DATA_ROOT, seq, "gt/gt.txt")

    seq_detections = parse_mot17_det(det_file)
    gt_df = parse_mot17_gt(gt_file)

    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

    acc = mm.MOTAccumulator(auto_id=True)

    # =========================
    # RESET BOTSORT TRACKER
    # =========================
    # (Đảm bảo biến 'args' đã được khai báo ở cell trước đó)
    tracker = BoTSORT(args, frame_rate=30)

    for k, img_path in enumerate(tqdm(image_paths, leave=False)):

        img = cv2.imread(img_path)
        if img is None:
            continue

        frame_dets = seq_detections.get(k, [])

        # -------- confidence filtering --------
        frame_dets = [d for d in frame_dets if d['score'] >= CONF_THRESH]

        dets_list = []
        for det in frame_dets:
            x1, y1, x2, y2 = det['boxs']
            score = det['score']
            dets_list.append([x1, y1, x2, y2, score])

        if len(dets_list) > 0:
            dets_array = np.array(dets_list, dtype=np.float32)
        else:
            dets_array = np.empty((0, 5), dtype=np.float32)

        # =========================
        # BOTSORT TRACKING UPDATE
        # =========================
        start_track = time.time()

        # Toàn bộ logic Predict, Camera Motion, Association được gom vào hàm update
        online_targets = tracker.update(dets_array, img)

        tracking_time += time.time() - start_track
        tracking_calls += 1

        # =========================
        # COLLECT HYPOTHESES
        # =========================
        frame_hypos = []
        frame_hypo_ids = []

        for t in online_targets:
            tlwh = t.tlwh # Định dạng chuẩn [x_top_left, y_top_left, w, h]
            frame_hypos.append([tlwh[0], tlwh[1], tlwh[2], tlwh[3]])
            frame_hypo_ids.append(t.track_id)

        # =========================
        # COLLECT GT
        # =========================
        frame_gt = []
        frame_gt_ids = []

        gt_now = gt_df[gt_df['frame'] == k]

        for _,row in gt_now.iterrows():
            frame_gt.append([row['x'], row['y'], row['w'], row['h']])
            frame_gt_ids.append(int(row['track_id']))

        dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
        acc.update(frame_gt_ids, frame_hypo_ids, dist_matrix)

    accs.append(acc)
    names.append(seq)


========== Running MOT17-02-DPM ==========


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


  0%|          | 0/600 [00:00<?, ?it/s]


========== Running MOT17-02-FRCNN ==========


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


...

Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


  0%|          | 0/750 [00:00<?, ?it/s]


========== Running MOT17-13-SDP ==========


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


  0%|          | 0/750 [00:00<?, ?it/s]

In [10]:
if tracking_calls > 0:
    print(f"Total tracker calls: {tracking_calls}")
    print(f"Total tracking time: {tracking_time:.3f} seconds")
    print(f"Average time per call: {(tracking_time/tracking_calls)*1000:.3f} ms")

Total tracker calls: 15948
Total tracking time: 1253.313 seconds
Average time per call: 78.587 ms


In [11]:
import os
import motmetrics as mm
from collections import defaultdict

mh = mm.metrics.create()

# Nhóm acc theo detector
detector_groups = defaultdict(list)

for name, acc in zip(names, accs):
    detector = name.split('-')[-1]  # DPM / FRCNN / SDP
    detector_groups[detector].append((name, acc))

for detector, items in detector_groups.items():
    print(f"\n========== RESULTS FOR DETECTOR: {detector} ==========\n")

    seq_names = [n for n, _ in items]
    seq_accs = [a for _, a in items]

    summary = mh.compute_many(
        seq_accs,
        names=seq_names,
        metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'],
        generate_overall=True
    )

    str_summary = mm.io.render_summary(summary)
    print(str_summary)


========== RESULTS FOR DETECTOR: DPM ==========

              num_frames      mota      motp      idf1       idp       idr  num_switches
MOT17-02-DPM         600  0.190632  0.230998  0.260054  0.727177  0.158340            50
MOT17-04-DPM        1050  0.310001  0.194971  0.404893  0.812066  0.269676           115
MOT17-05-DPM         837  0.312161  0.217579  0.412415  0.781424  0.280130            56
MOT17-09-DPM         525  0.493762  0.242906  0.506994  0.663181  0.410351            51
MOT17-10-DPM         654  0.299244  0.230560  0.346635  0.718378  0.228429            72
MOT17-11-DPM         900  0.505273  0.202670  0.547027  0.768997  0.424497            38
MOT17-13-DPM         750  0.095213  0.248044  0.161752  0.825219  0.089664            47
OVERALL             5316  0.292981  0.210307  0.377364  0.774535  0.249450           429

========== RESULTS FOR DETECTOR: FRCNN ==========

                num_frames      mota      motp      idf1       idp       idr  num_switches
MOT17-

In [12]:
import os

save_dir = "tracking_results"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "summary_last.csv")

summary.to_csv(save_path, float_format="%.4f")

print("Saved to:", save_path)

Saved to: tracking_results/summary_last.csv


In [13]:
# =========================
# LƯU KẾT QUẢ RA FILE
# =========================
# Tạo thư mục lưu trữ nếu chưa có (bạn có thể đổi tên thư mục tùy ý)
RESULT_ROOT = "mot17_results"
os.makedirs(RESULT_ROOT, exist_ok=True)

# 1. Lưu dạng Text (Giữ nguyên format căn lề đẹp như trên màn hình)
txt_path = os.path.join(RESULT_ROOT, "metrics_summary.txt")
with open(txt_path, "w") as f:
    f.write(str_summary)

# 2. Lưu dạng CSV (Dễ dàng mở bằng Excel, Google Sheets)
csv_path = os.path.join(RESULT_ROOT, "metrics_summary.csv")
summary.to_csv(csv_path)

print(f"\nĐã lưu báo cáo đánh giá vào: {txt_path} và {csv_path}")


Đã lưu báo cáo đánh giá vào: mot17_results/metrics_summary.txt và mot17_results/metrics_summary.csv
